# Create Delta table and test Primary constraint

- The source comes from jupyter-pyspark/f1-sourcefiles
- Create delta lake table with primary key
- Import circuits.csv file into dataframe
- Add an extra row to break primary key contraint
- Insert data into delta table


# Initalise a spark session

In [6]:
# Initalise a spark session
import os
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType,
    DoubleType, DateType, BooleanType
)
from pyspark.sql.functions import col
from delta.tables import DeltaTable

# Fix JAVA_HOME to your actual Java 21 path
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["PYSPARK_SUBMIT_ARGS"] = "--packages io.delta:delta-spark_2.12:3.2.0 pyspark-shell"

# Build Spark session with Delta Lake support
builder = SparkSession.builder \
    .appName("DeltaLakeExample") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()


# Setup schema (database)


In [7]:
spark.sql("CREATE DATABASE IF NOT EXISTS f1 COMMENT 'f1 schema'")
spark.sql("USE f1")


# races = spark.read.csv("f1-sourcefiles/races.csv", header=True, inferSchema=True).filter("year >= 2022")


# seasons = spark.read.csv("f1-sourcefiles/seasons.csv", header=True, inferSchema=True)

DataFrame[]

# Drop table if exists


In [34]:
spark.sql("DROP TABLE IF EXISTS f1.circuits")




DataFrame[]

# Use Alias in a RIGHT JOIN


In [35]:
# See where Spark is running from
print(f"Working directory: {os.getcwd()}")

# build an absolute path to it 

base_path  = os.getcwd()
table_path = f"{base_path}/delta/f1"

print(f"Writing to: {table_path}")


Working directory: /home/robyip/projects/pyspark-deltalake/jupyter-pyspark
Writing to: /home/robyip/projects/pyspark-deltalake/jupyter-pyspark/delta/f1


In [37]:
spark.sql(f"""
    CREATE TABLE f1.circuits (
        circuitId   INT       NOT NULL,
        circuitRef  STRING    NOT NULL,
        name        STRING    NOT NULL,
        location    STRING    NOT NULL,
        country     STRING,
        lat         DOUBLE,
        lng         DOUBLE,
        alt         INT       NOT NULL,
        url         STRING
    )
    USING DELTA
    LOCATION '{table_path}'
    COMMENT 'F1 circuits table'
    TBLPROPERTIES (
        'delta.autoOptimize.optimizeWrite' = 'true',
        'delta.autoOptimize.autoCompact'   = 'true'
    )
""")

DataFrame[]

# Make circuitId a PRIMARY KEY column
# Oops Primary key is not supported in open source version of Delta Lake 
# Something close to it

In [38]:
# 
# spark.sql(f"""
#    ALTER TABLE f1.circuits ADD CONSTRAINT pk_f1_circuits PRIMARY KEY (circuitId)
# """)

spark.sql(f"""
    ALTER TABLE f1.circuits ADD CONSTRAINT circuitId_not_null CHECK (circuitId IS NOT NULL)
""")

DataFrame[]

# Show the constraint is added

In [39]:
# Show table properties including constraints
spark.sql("SHOW TBLPROPERTIES f1.circuits").show(truncate=False)

+------------------------------------+---------------------+
|key                                 |value                |
+------------------------------------+---------------------+
|delta.autoOptimize.autoCompact      |true                 |
|delta.autoOptimize.optimizeWrite    |true                 |
|delta.constraints.circuitid_not_null|circuitId IS NOT NULL|
|delta.minReaderVersion              |1                    |
|delta.minWriterVersion              |3                    |
+------------------------------------+---------------------+



# Read the csv into a dataframe

In [40]:

 

df.show()

df.printSchema()


+---------+--------------+--------------------+------------+---------+--------+---------+---+--------------------+
|circuitId|    circuitRef|                name|    location|  country|     lat|      lng|alt|                 url|
+---------+--------------+--------------------+------------+---------+--------+---------+---+--------------------+
|        1|   albert_park|Albert Park Grand...|   Melbourne|Australia|-37.8497|  144.968| 10|http://en.wikiped...|
|        2|        sepang|Sepang Internatio...|Kuala Lumpur| Malaysia| 2.76083|  101.738| 18|http://en.wikiped...|
|        3|       bahrain|Bahrain Internati...|      Sakhir|  Bahrain| 26.0325|  50.5106|  7|http://en.wikiped...|
|        4|     catalunya|Circuit de Barcel...|    Montmeló|    Spain|   41.57|  2.26111|109|http://en.wikiped...|
|        5|      istanbul|       Istanbul Park|    Istanbul|   Turkey| 40.9517|   29.405|130|http://en.wikiped...|
|        6|        monaco|   Circuit de Monaco| Monte-Carlo|   Monaco| 43.7347| 

# Add a dummy row with a NOT NULL circuitId

In [41]:
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

# Use the same schema as your existing df
new_row = spark.createDataFrame([
    Row(circuitId=None, circuitRef="London", name="Test Circuit", location="London", country="UK", lat=51.5, lng=-0.1, alt=10, url="http://test.com")
], schema=df.schema)

df = df.unionByName(new_row)

df.show(df.count(), truncate=False)

+---------+--------------+-------------------------------------+---------------------+-------------+--------+---------+----+-----------------------------------------------------------------------+
|circuitId|circuitRef    |name                                 |location             |country      |lat     |lng      |alt |url                                                                    |
+---------+--------------+-------------------------------------+---------------------+-------------+--------+---------+----+-----------------------------------------------------------------------+
|1        |albert_park   |Albert Park Grand Prix Circuit       |Melbourne            |Australia    |-37.8497|144.968  |10  |http://en.wikipedia.org/wiki/Melbourne_Grand_Prix_Circuit              |
|2        |sepang        |Sepang International Circuit         |Kuala Lumpur         |Malaysia     |2.76083 |101.738  |18  |http://en.wikipedia.org/wiki/Sepang_International_Circuit              |
|3        |bahr

# This code removes the last row if I have added extra row by accident

In [44]:
df = df.limit(df.count() - 1)

df.show(df.count(), truncate=False)

+---------+--------------+-------------------------------------+---------------------+-------------+--------+---------+----+-----------------------------------------------------------------------+
|circuitId|circuitRef    |name                                 |location             |country      |lat     |lng      |alt |url                                                                    |
+---------+--------------+-------------------------------------+---------------------+-------------+--------+---------+----+-----------------------------------------------------------------------+
|1        |albert_park   |Albert Park Grand Prix Circuit       |Melbourne            |Australia    |-37.8497|144.968  |10  |http://en.wikipedia.org/wiki/Melbourne_Grand_Prix_Circuit              |
|2        |sepang        |Sepang International Circuit         |Kuala Lumpur         |Malaysia     |2.76083 |101.738  |18  |http://en.wikipedia.org/wiki/Sepang_International_Circuit              |
|3        |bahr

# Overwrite the data in the f1.circuits table

With a null circuitId row the insert will fail
Then remove the row with a null primary key - use the code above
And run the insert again - this should succeed


In [45]:
df.write.format("delta").mode("overwrite").saveAsTable("f1.circuits")

In [ ]:
# SELECT from the Delta tabke f1.circuits

In [48]:
spark.sql(f"""
   SELECT * FROM f1.circuits WHERE country = 'UK'
""").show()

+---------+------------+-------------------+----------------+-------+-------+--------+---+--------------------+
|circuitId|  circuitRef|               name|        location|country|    lat|     lng|alt|                 url|
+---------+------------+-------------------+----------------+-------+-------+--------+---+--------------------+
|        9| silverstone|Silverstone Circuit|     Silverstone|     UK|52.0786|-1.01694|153|http://en.wikiped...|
|       31|   donington|     Donington Park|Castle Donington|     UK|52.8306|-1.37528| 88|http://en.wikiped...|
|       38|brands_hatch|       Brands Hatch|            Kent|     UK|51.3569|0.263056|145|http://en.wikiped...|
|       58|     aintree|            Aintree|       Liverpool|     UK|53.4769|-2.94056| 20|http://en.wikiped...|
+---------+------------+-------------------+----------------+-------+-------+--------+---+--------------------+

